In [11]:
import pandas as pd
import numpy as np
from functools import reduce
import shutil, os

In [12]:
#Field, CDL, and GRIDMET ID Table
# this file is updated by outside of ET Demands workflow
#contains OpenET_ID, GRIDMET_ID, CDL_XXXX codes
cdl_data_path = r'C:\dri-owrd-et\tables\ee_exports\crop_type_codes_and_gridmet_cells.csv'

# read the csv into a pandas dataframe
cdl_df = pd.read_csv(cdl_data_path)


#%% replace cdl values with ETDemands values based on crosswalk
crosswalk_file = r'D:\owrd_etdemands\et_demands\OR_unique_cdl_etdemands_crosswalk_model_setup_1979_2024.csv'
crosswalk = pd.read_csv(crosswalk_file, index_col=0).squeeze('columns').to_dict()

cdl_subset = cdl_df.drop(['OPENET_ID'],axis='columns')
cdl_subset.set_index('GRIDMET_ID', inplace=True)

# get min max year in cdl list
header_names = cdl_subset.columns.tolist()
cdl_year_lst = [field[5:] for field in header_names]

min_cdl_yr = min(cdl_year_lst)
max_cdl_yr = max(cdl_year_lst)

print('CDL data includes {} to {}.'.format(min_cdl_yr, max_cdl_yr))


CDL data includes 1984 to 2024.


In [13]:
cdl_unique_values = sorted(pd.unique(cdl_subset.values.ravel()).tolist())
# print('Unique CDL Code: {}'.format(cdl_unique_values))


crosswalk_unique_cdl = sorted(list(crosswalk['etd_no'].keys()))
# print('Crosswalk CDL Codes: {}'.format(crosswalk_unique_cdl))


missing = list(set(cdl_unique_values)- set(crosswalk_unique_cdl))

if len(missing) > 0:
    print('Crosswalk file is missing CDL crop codes: {}'.format(list(missing)))
    print('Review crosswalk list before proceeding')
else:
    print('All CDL codes found in crosswalk list. Continue updating static files.')


All CDL codes found in crosswalk list. Continue updating static files.


In [6]:
print(cdl_unique_values)

[1, 4, 5, 6, 12, 13, 14, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 61, 63, 66, 67, 68, 69, 70, 71, 75, 76, 77, 82, 87, 111, 121, 122, 123, 124, 131, 141, 142, 143, 152, 176, 190, 195, 205, 206, 207, 208, 209, 210, 214, 216, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 236, 237, 241, 242, 243, 244, 246, 247, 249, 250]


In [14]:
cdl_subset.replace(crosswalk['etd_no'], inplace=True)

column_values = cdl_subset.values.ravel().astype(int)
unique_values =  pd.unique(column_values)


df_1 = cdl_subset.values.astype(int).tolist()

df_2 = pd.DataFrame(pd.Series(df_1))
df_2.set_index(cdl_subset.index, inplace=True)
df_2['id'] = df_2.index

grouped_df = df_2.groupby(['id'], as_index = False).agg({0: 'sum'})
grouped_df.set_index(grouped_df.id, inplace=True)

grouped_df2 = grouped_df.drop(['id'],axis='columns')

def drop_dup(lst):
    data = lst.tolist()
    flat_list = [item for sublist in data for item in sublist]
    # print(flat_list)
    res = []
    [res.append(x) for x in flat_list if x not in res]
    return res

final_cdl = grouped_df2.apply(lambda x: drop_dup(x), axis='columns')

# print(final_cdl.head())

# folder for new output
# directory_path = r'D:\owrd_etdemands\et_demands\{}_{}_static_files'.format(min_cdl_yr, max_cdl_yr)
# os.makedirs(directory_path, exist_ok=True)

static_directory_path = r'D:\owrd_etdemands\et_demands\static'

In [15]:
#%%
final_cdl.to_csv(r'{}\OR_etd_crop_list_per_gridcell_{}_{}_cdl.csv'.format(static_directory_path, min_cdl_yr, max_cdl_yr))

In [18]:
#note that this is the blank ET Demands static file with no rows. Script fills in unique rows based on CDL/gridMET data file

template_et_cells_path = r'C:\et-demands\et-demands\static\ETCellsCrops.txt'
# template_et_cells_path = r'D:\owrd_etdemands\et_demands\static\original_model_static\ETCellsCrops.txt'
shutil.copy(template_et_cells_path, static_directory_path)

# Rename the copied file
new_cells_name = r'ETCellsCrops_{}_{}.txt'.format(min_cdl_yr, max_cdl_yr)
print('Creating new ETCellsCrops.txt static file: '+ new_cells_name)
os.rename(r'{}\ETCellsCrops.txt'.format(static_directory_path),r'{}\{}'.format(static_directory_path, new_cells_name))

# read "new" file and write cell crops based on unique CDL gridcell list
cell_df_path = r'{}\{}'.format(static_directory_path, new_cells_name)
cell_df = pd.read_csv(template_et_cells_path, skiprows=[0,2], sep='\t')

cdl_cell_list_path = r'{}\OR_etd_crop_list_per_gridcell_{}_{}_cdl.csv'.format(static_directory_path, min_cdl_yr, max_cdl_yr)
cdl_df = pd.read_csv(cdl_cell_list_path)

cdl_df['id'] = cdl_df['id'].apply(lambda x: int(round(x, 0)))

# Write cell crops
# logging.debug('  {}'.format(cell_crops_path))
with open(cell_df_path, 'a') as output_f:
    for pos, row in cdl_df.iterrows():
        station_list = [row.id, row.id, row.id, 1]        
        crop_list = row['0'].replace('[','').replace(']','')
        crop_list = crop_list.split(",")
        crop_list = [int(x) for x in crop_list]

        crop_flag_list = [0] * 91
        
        for index in crop_list:
            # print(index)
            crop_flag_list[index-1] = 1
            
        output_list = station_list + crop_flag_list
        output_f.write('\t'.join(map(str, output_list)) + '\n')
        # break

Creating new ETCellsCrops.txt static file: ETCellsCrops_1984_2024.txt


In [19]:
# this is th original model file with information for all intersecting gridcells. We use this version and then filter to include all cells with active crops.
# The original OR ET Demands model grid (i.e. ET Cells grid) was built using an intersect with the field boundary .shp
# The intersect routine results in more ET Cells than the majority ("mode") field boundary assignments contain
# The ETCellsCrops and ETCellsProperties must contain the same unique grid cell count/ids
# This routine filters the original cell list to match the updated CDL/GridMET cell combinations
original_et_props_path = r'D:\owrd_etdemands\et_demands\static\original_model_static\ETCellsProperties.txt'
shutil.copy(original_et_props_path, static_directory_path)

# Rename the copied file
new_prop_name = r'ETCellsProperties_{}_{}.txt'.format(min_cdl_yr, max_cdl_yr)
print('Creating new ETCellsProperties.txt static file: '+ new_prop_name)
os.rename(r'{}\ETCellsProperties.txt'.format(static_directory_path),r'{}\{}'.format(static_directory_path, new_prop_name))


updated_et_props_path = r'{}\{}'.format(static_directory_path, new_prop_name)
cell_prop_df = pd.read_csv(updated_et_props_path, sep='\t')
unique_cells = cdl_df.id.unique()
filtered_df = cell_prop_df[cell_prop_df['ET Cell ID'].astype(int).isin(unique_cells.astype(int))]
filtered_df.to_csv(updated_et_props_path, index=False, sep ='\t')   

Creating new ETCellsProperties.txt static file: ETCellsProperties_1984_2024.txt
